# Technical indicators

Each indicator is one function in its own cell. Every function takes a DataFrame `df`
(columns `open, high, low, close, volume`, one row per candle, oldest first), calculates the
indicator for **every row**, adds the result as new column(s) to `df`, and returns `df`.
The first rows of an indicator are `NaN` until enough history exists.

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. SMA — Simple Moving Average
`SMA_t = (close_t + close_{t-1} + ... + close_{t-n+1}) / n`

In [31]:
def add_sma(df, period=20, column='close'):
    df[f'sma_{period}'] = df[column].rolling(window=period, min_periods=period).mean()
    return df

## 2. EMA — Exponential Moving Average
`k = 2 / (n + 1)`, `EMA_t = close_t * k + EMA_{t-1} * (1 - k)`, seeded with the SMA of the first `n` closes.

In [32]:
def add_ema(df, period=20, column='close'):
    values = df[column].to_numpy(dtype=float)
    ema = np.full(len(values), np.nan)
    if len(values) >= period:
        k = 2 / (period + 1)
        ema[period - 1] = values[:period].mean()
        for i in range(period, len(values)):
            ema[i] = values[i] * k + ema[i - 1] * (1 - k)
    df[f'ema_{period}'] = ema
    return df

## 3. DMA — Displaced Moving Average
An SMA shifted forward by `displacement` candles: `DMA_t = SMA_{t-d}`.

In [33]:
def add_dma(df, period=20, displacement=5, column='close'):
    sma = df[column].rolling(window=period, min_periods=period).mean()
    df[f'dma_{period}_{displacement}'] = sma.shift(displacement)
    return df

## 4. Bollinger Bands
`middle = SMA_n`, `upper = middle + k * std_n`, `lower = middle - k * std_n` (population standard deviation).

In [34]:
def add_bollinger_bands(df, period=20, num_std=2, column='close'):
    rolling = df[column].rolling(window=period, min_periods=period)
    middle = rolling.mean()
    std = rolling.std(ddof=0)
    df['bb_middle'] = middle
    df['bb_upper'] = middle + num_std * std
    df['bb_lower'] = middle - num_std * std
    df['bb_middle'] = middle
    return df

## 5. RSI — Relative Strength Index (Wilder)
`RS = avg_gain / avg_loss`, `RSI = 100 - 100 / (1 + RS)`.
First averages are simple means of the first `n` changes; afterwards `avg = (prev_avg * (n-1) + current) / n`.

In [35]:
def add_rsi(df, period=14, column='close'):
    change = df[column].diff().to_numpy(dtype=float)
    gain = np.where(change > 0, change, 0.0)
    loss = np.where(change < 0, -change, 0.0)
    rsi = np.full(len(df), np.nan)
    if len(df) > period:
        avg_gain = gain[1:period + 1].mean()
        avg_loss = loss[1:period + 1].mean()
        for i in range(period, len(df)):
            if i > period:
                avg_gain = (avg_gain * (period - 1) + gain[i]) / period
                avg_loss = (avg_loss * (period - 1) + loss[i]) / period
            if avg_loss == 0:
                rsi[i] = 100.0 if avg_gain > 0 else 50.0
            else:
                rsi[i] = 100 - 100 / (1 + avg_gain / avg_loss)
    df[f'rsi_{period}'] = rsi
    return df

## 6. Momentum
`Momentum_t = close_t - close_{t-n}`

In [36]:
def add_momentum(df, period=10, column='close'):
    df[f'momentum_{period}'] = df[column] - df[column].shift(period)
    return df

## 7. MACD — Moving Average Convergence Divergence
`MACD = EMA_fast - EMA_slow`, `signal = EMA_signal(MACD)`, `histogram = MACD - signal`.

In [37]:
def add_macd(df, fast=12, slow=26, signal=9, column='close'):
    fast_ema = df[column].ewm(span=fast, adjust=False, min_periods=fast).mean()
    slow_ema = df[column].ewm(span=slow, adjust=False, min_periods=slow).mean()
    macd = fast_ema - slow_ema
    macd_signal = macd.ewm(span=signal, adjust=False, min_periods=signal).mean()
    df['macd'] = macd
    df['macd_signal'] = macd_signal
    df['macd_histogram'] = macd - macd_signal
    return df

## 8. Volume
The raw `volume` column is the traded quantity per candle. This function validates it as numeric
and adds a volume moving average (`volume_ma_n`) plus relative volume (`volume / volume_ma_n`).

In [38]:
def add_volume(df, period=20):
    df['volume'] = pd.to_numeric(df['volume'], errors='raise')
    df[f'volume_ma_{period}'] = df['volume'].rolling(window=period, min_periods=period).mean()
    df[f'relative_volume_{period}'] = df['volume'] / df[f'volume_ma_{period}']
    return df

## 9. Volume delta
`volume_delta = buy_volume - sell_volume` (signed, not absolute).
If `buy_volume` / `sell_volume` are missing, it is estimated as `volume * (close - open) / (high - low)`,
which is negative for red candles and positive for green candles.

In [39]:
def add_volume_delta(df):
    if {'buy_volume', 'sell_volume'}.issubset(df.columns):
        df['volume_delta'] = df['buy_volume'] - df['sell_volume']
    else:
        candle_range = (df['high'] - df['low']).replace(0, np.nan)
        body_ratio = ((df['close'] - df['open']) / candle_range).fillna(0.0)
        df['volume_delta'] = df['volume'] * body_ratio
    return df

## 10. Ichimoku Cloud
- Tenkan-sen = (highest high + lowest low) / 2 over `tenkan` candles
- Kijun-sen = (highest high + lowest low) / 2 over `kijun` candles
- Senkou Span A = (Tenkan + Kijun) / 2, plotted `displacement` candles ahead
- Senkou Span B = (highest high + lowest low) / 2 over `senkou_b` candles, plotted `displacement` candles ahead
- Chikou Span = close plotted `displacement` candles back (uses future prices, so do not use it as an ML feature)

In [40]:
def add_ichimoku(df, tenkan=9, kijun=26, senkou_b=52, displacement=26):
    if displacement < 0:
        raise ValueError('Ichimoku displacement must be nonnegative.')
    def midpoint(period):
        high = df['high'].rolling(window=period, min_periods=period).max()
        low = df['low'].rolling(window=period, min_periods=period).min()
        return (high + low) / 2

    tenkan_sen = midpoint(tenkan)
    kijun_sen = midpoint(kijun)
    df['ichimoku_tenkan'] = tenkan_sen
    df['ichimoku_kijun'] = kijun_sen
    df['ichimoku_senkou_a'] = ((tenkan_sen + kijun_sen) / 2).shift(displacement)
    df['ichimoku_senkou_b'] = midpoint(senkou_b).shift(displacement)
    # Chikou is a chart back-plot; its negative shift is excluded from forecasting rows.
    return df

## 11. ATR — Average True Range
`TR_t = max(high_t - low_t, |high_t - close_{t-1}|, |low_t - close_{t-1}|)` (the first candle has no previous close, so it uses `high - low`).
- `method='wilder'` (default, the standard ATR): the first value is the simple mean of the first `n` TRs; afterwards `ATR_t = (ATR_{t-1} * (n-1) + TR_t) / n`. Column `atr_n`.
- `method='sma'`: plain rolling mean of the last `n` TRs. Column `atr_sma_n`. This is the version `regime_linreg.py` uses.

ATR is in price units and measures volatility (how far price typically moves per candle), not direction.

In [41]:
def add_atr(df, period=14, method='wilder'):
    prev_close = df['close'].shift(1)
    true_range = pd.concat([df['high'] - df['low'], (df['high'] - prev_close).abs(), (df['low'] - prev_close).abs()], axis=1).max(axis=1)
    true_range = true_range.to_numpy(dtype=float)  # first candle has no previous close -> high - low

    if method == 'sma':
        atr = pd.Series(true_range).rolling(window=period, min_periods=period).mean().to_numpy()
        name = f'atr_sma_{period}'
    elif method == 'wilder':
        atr = np.full(len(true_range), np.nan)
        if len(true_range) >= period:
            atr[period - 1] = true_range[:period].mean()
            for i in range(period, len(true_range)):
                atr[i] = (atr[i - 1] * (period - 1) + true_range[i]) / period
        name = f'atr_{period}'
    else:
        raise ValueError("method must be 'wilder' or 'sma'")

    df['true_range'] = true_range
    df[name] = atr
    return df

## Example: apply every indicator

In [42]:
from pathlib import Path
import argparse
import json
import re
import numpy as np
import pandas as pd

FEATURE_COLUMNS = [  # Indicator inputs
    "sma_20", "ema_8", "ema_20", "ema_21", "dma_20_5",
    "bb_middle", "bb_upper", "bb_lower", "rsi_14", "momentum_10",
    "macd", "macd_signal", "macd_histogram", "volume_ma_20",
    "relative_volume_20", "volume_delta", "ichimoku_tenkan",
    "ichimoku_kijun", "ichimoku_senkou_a", "ichimoku_senkou_b", "true_range", "atr_sma_14"
]


In [43]:
def add_all_indicators(df):  # Apply the complete agreed indicator set to one asset/timeframe.
    df = df.copy()  # Keep the caller's original frame unchanged.
    df = df.drop(columns=["ichimoku_chikou"], errors="ignore")  # Remove the known future-looking column if it already exists.
    df = add_sma(df, period=20)  # Add the twenty-bar average.
    for period in (8, 20, 21):  # Retain EMA20 and include the proposal's EMA8/EMA21 pair.
        df = add_ema(df, period=period)  # Calculate each EMA independently on this file's closes.
    df = add_dma(df, period=20, displacement=5)  # Add a genuinely displaced SMA rather than duplicating SMA20.
    df = add_bollinger_bands(df, period=20, num_std=2)  # Add three Bollinger columns.
    df = add_rsi(df, period=14)  # Add Wilder RSI.
    df = add_momentum(df, period=10)  # Add the ten-bar price difference.
    df = add_macd(df, fast=12, slow=26, signal=9)  # Add MACD, its signal, and histogram.
    df = add_volume(df, period=20)  # Add volume average and relative volume.
    df = add_volume_delta(df)  # Add or refresh delta and record its method.
    df = add_ichimoku(df)  # Add the four safe Ichimoku lines.
    df = add_atr(df, period=14, method="sma")  # Add the fourteen-bar ATR.
    return df  # Each row now contains all indicators available by that candle's close.


In [44]:
def candle_interval(path):  # Infer bar duration from the existing filename convention.
    match = re.search(r"_(\d+)([mhd])$", path.stem.lower())  # Recognize suffixes such as _5m, _1h, or _1d.
    if match is None:  # Duration is needed to distinguish the bar start from availability.
        raise ValueError(f"Missing timeframe suffix in {path.name}; expected _5m, _1h, etc.")  # Ask for an unambiguous filename.
    units = {"m": "minutes", "h": "hours", "d": "days"}  # Translate filename units to pandas units.
    interval = pd.Timedelta(**{units[match[2]]: int(match[1])})  # Construct that bar's nominal duration.
    if interval <= pd.Timedelta(0):  # Reject a meaningless zero-duration candle.
        raise ValueError("Candle interval must be positive.")  # Stop with a clear explanation.
    return interval  # Use this duration only for availability, not rolling window length.


In [45]:
def load_candles(path):  # Read and validate one input CSV without silently fixing its data.
    df = pd.read_csv(path)  # Keep all original columns.
    required = {"time", "open", "high", "low", "close", "volume"}  # Require the indicator inputs.
    missing = required - set(df.columns)  # Identify absent required columns.
    if missing or df.empty:  # Empty files and incomplete schemas need attention first.
        raise ValueError(f"Empty data or missing columns: {sorted(missing)}")  # Report the problem instead of inventing data.
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="raise")  # Normalize offset-aware times to UTC; naive values are assumed UTC.
    if df["time"].isna().any() or df["time"].duplicated().any():  # Ensure one valid time key per row.
        raise ValueError("Missing or duplicate timestamps; resolve before exporting indicators.")  # Avoid ambiguous alignment.
    df = df.sort_values("time", kind="stable").set_index("time")  # Use chronological UTC timestamps as row labels.
    numeric = ["open", "high", "low", "close", "volume"]  # Validate OHLCV even when CSV values were read as strings.
    numeric += [c for c in ("buy_volume", "sell_volume") if c in df.columns]  # Validate optional side volumes too.
    df[numeric] = df[numeric].apply(pd.to_numeric, errors="raise")  # Convert the selected columns to numbers.
    if not np.isfinite(df[numeric].to_numpy(dtype=float)).all():  # Reject missing or infinite numeric inputs.
        raise ValueError("OHLCV/side-volume inputs must be finite; clean missing values first.")  # Do not backfill from future candles.
    if (df[["open", "high", "low", "close"]] <= 0).any().any():  # These downloaded instruments require positive prices.
        raise ValueError("Found a zero or negative price.")  # Keep invalid prices out of recursive indicators.
    if (df[[c for c in numeric if c.endswith("volume")]] < 0).any().any():  # Recorded quantities cannot be negative.
        raise ValueError("Found negative volume.")  # Signed volume_delta is not included in this check.
    if (df["high"] < df[["open", "low", "close"]].max(axis=1)).any() or (df["low"] > df[["open", "high", "close"]].min(axis=1)).any():  # Check that the high and low contain the candle body.
        raise ValueError("Inconsistent OHLC prices.")  # Leave correction of bad source candles explicit.
    return df  # Return the validated frame with its time index.


In [46]:
def process_folder(input_dir, output_dir):  # Export all nested asset folders in a single run.
    input_dir, output_dir = Path(input_dir).resolve(), Path(output_dir).resolve()  # Normalize both folder paths.
    if input_dir == output_dir or input_dir in output_dir.parents or output_dir in input_dir.parents:  # Keep the input and output trees separate.
        raise ValueError("Choose separate, non-nested input and output folders.")  # Prevent reading generated CSVs or replacing source files.
    files = sorted(input_dir.rglob("*.csv"))  # Snapshot all source CSV paths before writing anything.
    if not files:  # A wrong input path should not look like a successful export.
        raise FileNotFoundError(f"No CSV files found under {input_dir}")  # Ask for the folder containing the downloaded data.
    report = []  # Collect one success or failure record per file.
    for source in files:  # Process each asset/timeframe independently.
        relative = source.relative_to(input_dir)  # Preserve the asset subfolder and filename.
        try:  # Record a bad file without preventing valid files from exporting.
            interval = candle_interval(source)  # Determine this file's nominal bar length.
            df = load_candles(source)  # Read, sort, and validate its own candles.
            df = add_all_indicators(df)  # Calculate the ten indicator families on the full chronological history.
            df["available_at"] = df.index + interval  # The source time is the bar start; inputs become known at its nominal end.
            if (df["available_at"] > pd.Timestamp.now(tz="UTC")).any():  # Reject bars whose nominal close is still in the future.
                raise ValueError("Contains a not-yet-closed candle; supply completed historical bars.")  # Do not export an unstable live candle.
            if np.isinf(df[FEATURE_COLUMNS].to_numpy(dtype=float)).any():  # Allow expected warm-up NaNs, but not infinity.
                raise ValueError("An indicator produced infinity.")  # Stop this file rather than hide a calculation problem.
            destination = output_dir / relative  # Recreate the same relative location in the output tree.
            destination.parent.mkdir(parents=True, exist_ok=True)  # Create the output asset folder when needed.
            temporary = destination.with_suffix(".csv.tmp")  # Write a complete new file before replacing an older output.
            df.to_csv(temporary, index=True, index_label="time")  # Save original values and all added columns; NaNs become blank fields.
            temporary.replace(destination)  # Publish the completed output file atomically.
            ready = int(df[FEATURE_COLUMNS].notna().all(axis=1).sum())  # Count rows with every indicator available.
            report.append({"file": str(relative), "status": "ok", "rows": len(df), "all_indicators_ready": ready, "columns": len(df.columns) + 1})  # Record an auditable summary.
            print(f"Saved {relative}: {len(df):,} rows; {ready:,} fully warmed up.", flush=True)  # Show progress during larger exports.
        except Exception as exc:  # Keep both input problems and export failures visible.
            report.append({"file": str(relative), "status": "error", "error": str(exc)})  # Record the exact failed file and reason.
            print(f"FAILED {relative}: {exc}", flush=True)  # Clearly distinguish failures from successful files.
    output_dir.mkdir(parents=True, exist_ok=True)  # Ensure the report can be saved even if every input failed.
    manifest = {"feature_columns": FEATURE_COLUMNS, "time_convention": "time=candle_start_UTC; available_at=nominal_candle_end_UTC", "files": report}  # Document the model-input whitelist and timing convention.
    (output_dir / "_indicator_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")  # Save a JSON report without creating another input-like CSV.
    if any(row["status"] == "error" for row in report):  # Do not describe a partial batch as a complete success.
        raise RuntimeError("Some files failed; inspect _indicator_manifest.json. Successful exports remain available.")  # Make unresolved files explicit.
    return pd.DataFrame(report)  # Display a compact per-file table in the notebook.


In [47]:
INPUT_DIR = Path("datapoints")  # Set this to your existing folder of downloaded asset folders.
OUTPUT_DIR = Path("datapoints_indicators")  # Choose a separate output folder for enriched copies.
summary = process_folder(INPUT_DIR, OUTPUT_DIR)  # Calculate and save every CSV, one file at a time.
summary  # Display row counts, success status, and fully warmed-up rows for each file.


Saved AAPL.US\AAPL.US_15m.csv: 62,784 rows; 62,707 fully warmed up.
Saved AAPL.US\AAPL.US_1h.csv: 16,973 rows; 16,896 fully warmed up.
Saved AAPL.US\AAPL.US_1m.csv: 939,806 rows; 939,729 fully warmed up.
Saved AAPL.US\AAPL.US_30m.csv: 31,440 rows; 31,363 fully warmed up.
Saved AAPL.US\AAPL.US_4h.csv: 5,682 rows; 5,605 fully warmed up.
Saved AAPL.US\AAPL.US_5m.csv: 188,117 rows; 188,040 fully warmed up.
Saved AMD.US\AMD.US_15m.csv: 58,228 rows; 58,151 fully warmed up.
Saved AMD.US\AMD.US_1h.csv: 15,684 rows; 15,607 fully warmed up.
Saved AMD.US\AMD.US_1m.csv: 872,108 rows; 872,031 fully warmed up.
Saved AMD.US\AMD.US_30m.csv: 29,119 rows; 29,042 fully warmed up.
Saved AMD.US\AMD.US_4h.csv: 5,253 rows; 5,176 fully warmed up.
Saved AMD.US\AMD.US_5m.csv: 174,603 rows; 174,526 fully warmed up.
Saved AMZN.US\AMZN.US_15m.csv: 62,830 rows; 62,753 fully warmed up.
Saved AMZN.US\AMZN.US_1h.csv: 16,977 rows; 16,900 fully warmed up.
Saved AMZN.US\AMZN.US_1m.csv: 940,260 rows; 940,183 fully warmed 

,file,status,rows,all_indicators_ready,columns
0,AAPL.US\AAPL.US_15m.csv,ok,62784,62707,30
1,AAPL.US\AAPL.US_1h.csv,ok,16973,16896,30
2,AAPL.US\AAPL.US_1m.csv,ok,939806,939729,30
3,AAPL.US\AAPL.US_30m.csv,ok,31440,31363,30
4,AAPL.US\AAPL.US_4h.csv,ok,5682,5605,30
...,...,...,...,...,...
91,XAU\XAU_1h.csv,ok,63814,63737,30
92,XAU\XAU_1m.csv,ok,3822306,3821936,30
93,XAU\XAU_30m.csv,ok,127557,127480,30
94,XAU\XAU_4h.csv,ok,17255,17178,30


In [50]:
example_path = OUTPUT_DIR / summary.iloc[0]["file"]  # Choose the first successfully processed file.
preview = pd.read_csv(example_path, nrows=82)  # Read just enough rows to see the full warm-up boundary.
preview.loc[0:81, ["time", "available_at", "ema_8", "ema_21", "rsi_14", "ichimoku_senkou_b", "true_range"]]  # Inspect timestamp alignment and warm-up values.


,time,available_at,ema_8,ema_21,rsi_14,ichimoku_senkou_b,true_range
0,2017-02-28 15:15:00+00:00,2017-02-28 15:30:00+00:00,NaN,NaN,NaN,NaN,0.0445
1,2017-02-28 15:30:00+00:00,2017-02-28 15:45:00+00:00,NaN,NaN,NaN,NaN,0.0600
2,2017-02-28 15:45:00+00:00,2017-02-28 16:00:00+00:00,NaN,NaN,NaN,NaN,0.0347
3,2017-02-28 16:00:00+00:00,2017-02-28 16:15:00+00:00,NaN,NaN,NaN,NaN,0.0278
4,2017-02-28 16:15:00+00:00,2017-02-28 16:30:00+00:00,NaN,NaN,NaN,NaN,0.0705
...,...,...,...,...,...,...,...
77,2017-03-02 21:00:00+00:00,2017-03-02 21:15:00+00:00,34.747993,34.796237,39.124006,34.62,0.0062
78,2017-03-03 14:30:00+00:00,2017-03-03 14:45:00+00:00,34.758995,34.796351,49.132942,34.62,0.1690
79,2017-03-03 14:45:00+00:00,2017-03-03 15:00:00+00:00,34.764329,34.795138,47.279717,34.62,0.0780
80,2017-03-03 15:00:00+00:00,2017-03-03 15:15:00+00:00,34.776589,34.797352,52.170320,34.62,0.0800
